<a href="https://colab.research.google.com/github/marcelovolta/sr_uba_2026/blob/main/suprise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Claude explanation
Older Surprise builds broke with NumPy 2, so the original notebook downgraded NumPy. Surprise 1.1.5 works with NumPy 2.x, so this cell is now commented out.

In [1]:
# Claude fix: not needed; scikit-surprise 1.1.5 works with NumPy 2.x, and NumPy 1.26.4 has no wheels for Python 3.13
# # ! pip install "numpy<2" --upgrade
# ! pip uninstall numpy -y
# ! pip install numpy==1.26.4
# ## restart session

### Claude explanation
Shows the installed NumPy version. Any 2.x version is fine with Surprise 1.1.5.

In [2]:
import numpy as np
np.__version__

'2.5.4'

In [5]:
! pip install pandas --quiet
! pip install --quiet scikit-surprise

In [6]:
import surprise as sp
import pandas as pd
import sqlite3

### Claude explanation
Loads the reader–book interactions from SQLite, keeping only books and readers with more than 20 ratings each. This reduces sparsity and makes the comparison between algorithms more meaningful.

In [ ]:
# conn = sqlite3.connect('data.db')
conn = sqlite3.connect('datos/datos.db')  # Claude fix: the DB is in datos/; a wrong path silently creates an empty DB

df = pd.read_sql("""
    SELECT *
      FROM interacciones
     WHERE id_libro IN (SELECT id_libro FROM interacciones GROUP BY 1 HAVING count(*) > 20)
       AND id_lector IN (SELECT id_lector FROM interacciones GROUP BY 1 HAVING count(*) > 20)
     """, conn)

print(df.shape)
df.head()

(293307, 4)


,id_lector,id_libro,fecha,rating
0,05-03-1970,1984,18-12-2018,9
1,05-03-1970,al-sur-de-la-frontera-al-oeste-del-sol,19-12-2018,9
2,05-03-1970,ampliacion-del-campo-de-batalla,14-11-2019,8
3,05-03-1970,amsterdam,18-12-2018,7
4,05-03-1970,apegos-feroces,25-03-2024,9


### Claude explanation
Converts the DataFrame into a Surprise `dataset`:
- `Reader` declares the rating scale (1–10).
- Columns must be in the order **user, item, rating**.
- 80/20 train/test split with a fixed seed so the algorithms can be compared.

In [8]:
reader = sp.reader.Reader(rating_scale=(1, 10))

data = sp.dataset.Dataset.load_from_df(df[['id_lector', 'id_libro', 'rating']], reader) # cuidado con el orden
trainset, testset = sp.model_selection.train_test_split(data, test_size=0.2, random_state=42)

# Algoritmos de predicción

## BaselineOnly

### Claude explanation
**BaselineOnly** predicts `global mean + user bias + item bias` and ignores interactions between users and items. It is the reference RMSE the other models should beat.

In [9]:
algo = sp.prediction_algorithms.baseline_only.BaselineOnly(verbose=False)
algo.fit(trainset)

predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

RMSE: 1.5304
RMSE: 1.5303548355668115


## KNN

### Claude explanation
**KNNBasic**, user-based: predicts a rating as the similarity-weighted average of the ratings of the `k=40` most similar users (Pearson) who rated the item. If fewer than `min_k=5` neighbors are found, it falls back to the global mean.

In [10]:
sim_options = {
    'name': 'pearson', #'cosine', 'pearson_baseline'
    'user_based': True, #False,
}

algo = sp.prediction_algorithms.knns.KNNBasic(k=40, min_k=5, sim_options=sim_options, random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.6327
RMSE: 1.632745116630586


### Claude explanation
**KNNWithMeans**: same as above, but uses each neighbor's deviation from their own mean rating. This corrects for users who systematically rate high or low.

In [11]:
sim_options = {
    'name': 'pearson', #'cosine', 'pearson_baseline'
    'user_based': True, #False,
}

algo = sp.prediction_algorithms.knns.KNNWithMeans(k=40, min_k=5, sim_options=sim_options, random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.5263
RMSE: 1.526337011740044


## SVD

### Claude explanation
**SVD** (Funk-style matrix factorization): learns latent factor vectors for users and items, plus biases, using SGD. The prediction is `mean + biases + dot(user, item)`.

In [12]:
algo = sp.prediction_algorithms.matrix_factorization.SVD(n_factors=500, n_epochs=20, random_state=42)

algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

RMSE: 1.5190
RMSE: 1.5189833603253826


## NMF

### Claude explanation
**NMF**: matrix factorization with non-negative factors and no bias terms. It usually performs worse than SVD on explicit ratings, and with 500 factors it is prone to overfitting.

In [13]:
algo = sp.prediction_algorithms.matrix_factorization.NMF(n_factors=500, n_epochs=20, random_state=42)

algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

RMSE: 3.2094
RMSE: 3.2093722196810233


# Predicción

### Claude explanation
Production stage: reloads **all** interactions without the >20 filter, so that every reader and book can get a prediction.

In [14]:
%%time
# conn = sqlite3.connect("data.db")
conn = sqlite3.connect("datos/datos.db")  # Claude fix: the DB is in datos/; a wrong path silently creates an empty DB
df = pd.read_sql('SELECT * FROM interacciones;', conn)

reader = sp.reader.Reader(rating_scale=(1, 10))
data = sp.dataset.Dataset.load_from_df(df[['id_lector', 'id_libro', 'rating']], reader) # cuidado con el orden

CPU times: user 447 ms, sys: 25.8 ms, total: 473 ms
Wall time: 486 ms


### Claude explanation
Trains the chosen model (BaselineOnly) on the full dataset. There is no test split here because the goal is to serve predictions, not to evaluate.

In [15]:
%%time

# algoritmo
algo = sp.prediction_algorithms.baseline_only.BaselineOnly(verbose=False)
algo.fit(data.build_full_trainset())

CPU times: user 471 ms, sys: 12.1 ms, total: 483 ms
Wall time: 482 ms


### Claude explanation
Saves the trained model to disk and deletes it from memory, to simulate loading it later in a separate service.

In [16]:
sp.dump.dump("algo.pkl", algo=algo)
del algo

In [17]:
_, algo = sp.dump.load("algo.pkl")

### Claude explanation
Predicts a reader's rating for specific books. The third argument (`r_ui`) is the actual rating, if known, and is only shown for reference. `.est` is the predicted value. For an unknown user or book, the bias is set to 0, so the prediction moves toward the global mean.

In [18]:
lector_id = "popocito"

libro_id = "reina-roja"
print(libro_id, algo.predict(lector_id, libro_id, 9))
print(libro_id, algo.predict(lector_id, libro_id, 9).est)

libro_id = "la-novia-rebelde"
print(libro_id, algo.predict(lector_id, libro_id, 7).est)

reina-roja user: popocito   item: reina-roja r_ui = 9.00   est = 6.74   {'was_impossible': False}
reina-roja 6.739325642987732
la-novia-rebelde 6.6702463594346
